<centre>$$ Transformer Block$$</centre>

bolck is not possable bcuz you make full transformer then i will work so 

In [ ]:
import torch 
import torch.nn as nn
import torch.nn.functional as F 
import math
#_____________________________________________------attention 
def scaled_dot_product_attention(Q,K,V, mask=None):
    d_k = Q.size(-1)
    scores = torch.matmul(Q,K.tranpose(-2,-1))
    scores = scores/math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0,-1e9)
        weights = F.softmax(scores,dim=-1)
        output = torch.matmul(weights,V)
        return output, weights
#class multi head attention 
class MultiHeadAtention(nn.Module):
    def __init__(self, d_model,num_heads, dropout = 0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads 
        self.d_k = d_model // num_heads
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model,d_model)
        self.dropout = nn.Dropout(dropout)
    def split_heads(self, x,batch):
        x = x.veiw(batch,-1,self.num_heads,self.d_k)
        return x.transpose(1,2)
    def forward (self,Q,K,V, mask=None):
        batch = Q.size(0)
        
        Q = self.split_heads(self.W_Q(Q), batch)
        K = self.split_heads(self.W_K(K), batch)
        V = self.split_heads(self.W_V(V), batch)
        out, weigths = scaled_dot_product_attention(
            Q,K,V,mask
        )
        out = out.transpose(1,2).contiguous()
        out = out.view(batch,-1,self.d_model)
        out = self.W_O(out)
        return out,weigths
class feedforward(nn.Module):
    def __init__(self,d_model,d_ff,dropout= 0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model,d_ff)
        self.linear2 = nn.Linear(d_ff,d_model)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()
    def forward(self,x ):
        x = self.relu(self.linear1(x))
        x = self.dropout(x)
        x = self.linear2(x)
        return x
class Postionalencoding(nn.Module):
    def __init__(self,d_model,max_len=5000,dropout = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len,d_model)
        pos = torch.arange(0,max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0,d_model,2).float() *(-math.log(10000.0/d_model)
        ))
        pe[:, 0::2] = torch.sin(pos * div_term)
        pe[:, 1::2] = torch.cos(pos * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)


